# Local Genius Pop seed export

로컬 Jupyter에서 Genius Pop 태그의 곡과 가사를 수집해 `genius_pop_seed.json`까지만 생성합니다. YouTube 다운로드는 수행하지 않습니다.

## 0. 패키지 설치

In [ ]:
%pip install -q lyricsgenius

## 1. 출력 경로와 수집 범위

In [ ]:
from pathlib import Path

OUTPUT_PATH = Path.cwd() / 'genius_pop_seed.json'
GENIUS_TAG = 'pop'
MAX_TRACKS = 500
MAX_PAGES = 100
REQUEST_DELAY_SECONDS = 1.0
DEFAULT_LANGUAGE = 'English'

print('output:', OUTPUT_PATH.resolve())
print('target tracks:', MAX_TRACKS)

## 2. Genius access token 입력

Genius API client access token을 입력합니다. 입력값은 화면에 표시되거나 JSON에 저장되지 않습니다.

In [ ]:
import os
from getpass import getpass

GENIUS_ACCESS_TOKEN = os.environ.get('GENIUS_ACCESS_TOKEN') or getpass(
    'GENIUS_ACCESS_TOKEN: '
)
assert GENIUS_ACCESS_TOKEN, 'GENIUS_ACCESS_TOKEN이 필요합니다.'
print('Genius token loaded')

## 3. Genius Pop seed 생성

Genius API의 tag endpoint를 사용합니다. 곡 하나가 완료될 때마다 JSON을 저장하며, 셀을 다시 실행하면 기존 곡을 건너뛰고 이어서 수집합니다.

In [ ]:
import json
import re
import time
from collections.abc import Mapping
from datetime import datetime, timezone
from urllib.parse import urlparse

import lyricsgenius

def write_seed(path, payload):
    payload['metadata'].update({
        'updated_at': datetime.now(timezone.utc).isoformat(),
        'num_tracks': len(payload['tracks']),
        'num_errors': len(payload['errors']),
    })
    path = Path(path).expanduser().resolve()
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    temporary_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    temporary_path.replace(path)

def load_seed(path):
    path = Path(path).expanduser().resolve()
    if path.is_file():
        payload = json.loads(path.read_text(encoding='utf-8'))
        if isinstance(payload.get('tracks'), list) and isinstance(payload.get('errors'), list):
            payload.setdefault('metadata', {})
            return payload
    return {
        'metadata': {
            'name': 'mohim_genius_seed',
            'tag': GENIUS_TAG,
            'generated_at': datetime.now(timezone.utc).isoformat(),
            'interrupted': False,
        },
        'tracks': [],
        'errors': [],
    }

def artist_label(value):
    if isinstance(value, Mapping):
        return str(value.get('name') or value.get('artist_name') or '').strip()
    return str(value or '').strip()

def iter_tag_candidates(client, tag, max_pages):
    page = 1
    seen = set()
    while page and page <= max_pages:
        response = client.tag(tag, page=page) or {}
        hits = response.get('hits') or []
        print(f'[Genius page {page}] candidates={len(hits)}', flush=True)
        for hit in hits:
            genius_url = str(hit.get('url') or '').strip()
            title = str(hit.get('title') or '').strip()
            artists = hit.get('artists') or []
            if isinstance(artists, (str, Mapping)):
                artists = [artists]
            artist = ', '.join(dict.fromkeys(
                name for name in (artist_label(value) for value in artists) if name
            ))
            if not artist:
                title_with_artists = str(hit.get('title_with_artists') or '')
                if ' by ' in title_with_artists:
                    _, artist = title_with_artists.rsplit(' by ', 1)
                    artist = artist.strip()
            if genius_url and title and artist and genius_url not in seen:
                seen.add(genius_url)
                yield {
                    'artist': artist,
                    'title': title,
                    'genius_url': genius_url,
                }
        next_page = response.get('next_page')
        page = int(next_page) if next_page else 0

def clean_lyrics(value):
    lyrics = str(value or '').strip()
    lyrics = re.sub(r'^.*?Lyrics\s*', '', lyrics, count=1, flags=re.I | re.S)
    return re.sub(r'\s*\d*Embed\s*$', '', lyrics, flags=re.I).strip()

genius = lyricsgenius.Genius(
    GENIUS_ACCESS_TOKEN,
    verbose=False,
    remove_section_headers=False,
    skip_non_songs=True,
    excluded_terms=['(Remix)', '(Live)'],
)
payload = load_seed(OUTPUT_PATH)
payload['metadata']['tag'] = GENIUS_TAG
payload['metadata']['interrupted'] = False
existing_urls = {
    str(track.get('lyrics_source_url') or track.get('candidate_url') or '')
    for track in payload['tracks']
}
write_seed(OUTPUT_PATH, payload)
print(f'resume tracks: {len(payload["tracks"])}', flush=True)

try:
    for attempted, candidate in enumerate(
        iter_tag_candidates(genius, GENIUS_TAG, MAX_PAGES),
        1,
    ):
        if len(payload['tracks']) >= MAX_TRACKS:
            break
        genius_url = candidate['genius_url']
        if genius_url in existing_urls:
            continue
        try:
            lyrics = clean_lyrics(genius.lyrics(song_url=genius_url))
            if not lyrics:
                raise RuntimeError('Genius returned no lyrics')
            slug = urlparse(genius_url).path.strip('/').removesuffix('-lyrics')
            payload['tracks'].append({
                'id': slug,
                'artist': candidate['artist'],
                'title': candidate['title'],
                'genres': [GENIUS_TAG.title()],
                'language': DEFAULT_LANGUAGE,
                'lyrics': lyrics,
                'lyrics_source': 'genius',
                'lyrics_source_url': genius_url,
                'candidate_url': genius_url,
                'candidate_source': f'genius_tag:{GENIUS_TAG}',
            })
            existing_urls.add(genius_url)
            status = f'saved ({len(payload["tracks"])}/{MAX_TRACKS})'
        except Exception as exc:
            payload['errors'].append({
                **candidate,
                'error': f'{type(exc).__name__}: {exc}',
            })
            status = f'failed: {type(exc).__name__}: {exc}'
        write_seed(OUTPUT_PATH, payload)
        print(
            f'[{attempted}] {candidate["artist"]} - {candidate["title"]}: {status}',
            flush=True,
        )
        if REQUEST_DELAY_SECONDS:
            time.sleep(REQUEST_DELAY_SECONDS)
except KeyboardInterrupt:
    payload['metadata']['interrupted'] = True
    write_seed(OUTPUT_PATH, payload)
    print('interrupted; completed tracks remain saved', flush=True)

write_seed(OUTPUT_PATH, payload)
print(json.dumps(payload['metadata'], ensure_ascii=False, indent=2))

## 4. 결과 확인

생성된 JSON을 Google Drive에 업로드한 뒤 Colab의 YouTube 다운로드 단계에서 사용합니다.

In [ ]:
import pandas as pd

seed_result = json.loads(OUTPUT_PATH.read_text(encoding='utf-8'))
print('seed JSON:', OUTPUT_PATH.resolve())
print('tracks:', len(seed_result['tracks']))
print('errors:', len(seed_result['errors']))
display(pd.DataFrame(seed_result['tracks'])[['artist', 'title', 'lyrics_source_url']].head(20))